In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
!pip install seqeval

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seqeval", "scikit-learn", "nltk"])

/kaggle/input/datasets/wantodbz/ncbi-disease/validation.csv
/kaggle/input/datasets/wantodbz/ncbi-disease/train.csv
/kaggle/input/datasets/wantodbz/ncbi-disease/test.csv
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=a4431a3f378ef45ed0fec73ac30dafdb06296153f0a68925d37a2316c38ace00
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


0

In [ ]:
# === CELL BREAK ===
import pandas as pd
import numpy as np
import ast, re, random, warnings, time, os
from collections import defaultdict
from copy import deepcopy

from seqeval.metrics import (
    f1_score as seqeval_f1,
    precision_score as seqeval_p,
    recall_score as seqeval_r,
)

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForTokenClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

os.environ["HF_TOKEN"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [ ]:
# --- KONFIGURASI ---
MODEL_NAME    = 'dmis-lab/biobert-v1.1'
NUM_EPOCHS    = 5
LEARNING_RATE = 5e-5
BATCH_SIZE    = 8
MAX_SEQ_LEN   = 512

TAG2IDX = {'O': 0, 'B-Disease': 1, 'I-Disease': 2}
IDX2TAG = {v: k for k, v in TAG2IDX.items()}
NUM_TAGS = len(TAG2IDX)

GRID_SEED     = 64
SAMPLING_SEED = 64

FEW_SHOT_FRACTION = 0.02
FEW_SHOT_LABEL    = '2%'

K_GRID = [1, 2, 3, 4, 5, 6, 7, 8, 9]
RS_FIXED = 15   # fixed, tidak digunakan di EXP 3

OUTPUT_DIR = '/kaggle/working/gridsearch_k_ncbi'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset       : NCBI Disease")
print(f"Scenario      : {FEW_SHOT_LABEL}")
print(f"Grid search   : K ∈ {K_GRID}")
print(f"RS (fixed)    : {RS_FIXED} (tidak aktif di EXP 3)")
print(f"Seed          : {GRID_SEED}")
print(f"Epochs        : {NUM_EPOCHS}, LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}")

Dataset       : NCBI Disease
Scenario      : 2%
Grid search   : K ∈ [1, 2, 3, 4, 5, 6, 7, 8, 9]
RS (fixed)    : 15 (tidak aktif di EXP 3)
Seed          : 64
Epochs        : 5, LR: 5e-05, Batch: 8


In [4]:
# --- UTILITIES ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def parse_sentences(df):
    sentences = []
    for idx, row in df.iterrows():
        tokens_str = row['tokens'].replace("'", '"')
        tokens_str = re.sub(r'\s+', ', ', tokens_str[1:-1])
        tokens_str = '[' + tokens_str + ']'
        try:
            tokens = ast.literal_eval(tokens_str)
        except:
            tokens = re.findall(r"'([^']*)'", row['tokens'])
        tags_str = row['ner_tags'].replace('[', '').replace(']', '')
        tags = [int(t) for t in tags_str.split()]
        iob_tags = []
        for tag in tags:
            if tag == 0:   iob_tags.append('O')
            elif tag == 1: iob_tags.append('B-Disease')
            elif tag == 2: iob_tags.append('I-Disease')
            else:          iob_tags.append('O')
        if len(tokens) == len(iob_tags):
            sentences.append({'id': row['id'], 'tokens': tokens, 'tags': iob_tags})
    return sentences

def sample_few_shot(sentences, fraction, seed=SAMPLING_SEED):
    rng = random.Random(seed)
    n   = max(1, int(len(sentences) * fraction))
    idx = list(range(len(sentences)))
    rng.shuffle(idx)
    return [sentences[i] for i in idx[:n]]

# === CELL BREAK ===
# --- COSINER ---
def extract_mentions_from_sentences(sentences):
    mention_contexts = defaultdict(list)
    mention_types    = {}
    for sent in sentences:
        tokens, tags = sent['tokens'], sent['tags']
        i = 0
        while i < len(tags):
            if tags[i].startswith('B-'):
                start, etype = i, tags[i][2:]
                i += 1
                while i < len(tags) and tags[i].startswith('I-'): i += 1
                mention = " ".join(tokens[start:i])
                mention_contexts[mention].append((tokens, start, i))
                mention_types[mention] = etype
            else: i += 1
    return mention_contexts, mention_types

def build_entity_embeddings(mention_contexts, model_name=MODEL_NAME):
    emb_device    = "cuda" if torch.cuda.is_available() else "cpu"
    emb_tokenizer = AutoTokenizer.from_pretrained(model_name)
    emb_model     = AutoModel.from_pretrained(model_name).to(emb_device)
    emb_model.eval()
    embeddings = {}
    for mention, contexts in mention_contexts.items():
        c = len(contexts); lr = 1.0 / c; v_concept = None
        for tokens, start, end in contexts:
            sentence = " ".join(tokens)
            inputs   = emb_tokenizer(sentence, return_tensors="pt",
                                     truncation=True, max_length=512)
            inputs   = {k: v.to(emb_device) for k, v in inputs.items()}
            with torch.no_grad():
                hidden = emb_model(**inputs).last_hidden_state[0]
            v_context = hidden[1:-1].mean(dim=0).cpu().numpy()
            if v_concept is None: v_concept = v_context.copy()
            else:
                norm_prod = np.linalg.norm(v_concept) * np.linalg.norm(v_context) + 1e-8
                sim       = max(0, np.dot(v_concept, v_context) / norm_prod)
                v_concept = v_concept + lr * (1 - sim) * v_context
        embeddings[mention] = v_concept
    del emb_model; torch.cuda.empty_cache()
    return embeddings

def compute_similarity_rankings(embeddings, mention_types):
    mentions = list(embeddings.keys())
    if len(mentions) < 2: return {}
    emb_matrix = np.array([embeddings[m] for m in mentions])
    sim_mat    = cosine_similarity(emb_matrix)
    rankings   = {}
    for i, mention in enumerate(mentions):
        mtype  = mention_types.get(mention, "")
        scores = [(mentions[j], sim_mat[i][j])
                  for j in range(len(mentions))
                  if i != j and mention_types.get(mentions[j], "") == mtype]
        scores.sort(key=lambda x: x[1], reverse=True)
        rankings[mention] = scores
    return rankings

def augment_sentence(sent, rankings, k):
    tokens, tags = sent['tokens'], sent['tags']
    mentions = []
    i = 0
    while i < len(tags):
        if tags[i].startswith('B-'):
            start = i; etype = tags[i][2:]; i += 1
            while i < len(tags) and tags[i].startswith('I-'): i += 1
            mentions.append((start, i, " ".join(tokens[start:i]), etype))
        else: i += 1
    if not mentions: return []
    augmented = []
    for aug_idx in range(k):
        new_tokens, new_tags = list(tokens), list(tags)
        n_rep = 0
        current_mentions = list(mentions)
        for idx_m, (start, end, mention_text, etype) in enumerate(current_mentions):
            if mention_text not in rankings: continue
            sims = rankings[mention_text]
            if aug_idx >= len(sims): continue
            replacement, _ = sims[aug_idx]
            rep_tok  = replacement.split()
            rep_tags = ['B-' + etype] + ['I-' + etype] * (len(rep_tok) - 1)
            new_tokens = new_tokens[:start] + rep_tok  + new_tokens[end:]
            new_tags   = new_tags[:start]   + rep_tags + new_tags[end:]
            offset = len(rep_tok) - (end - start)
            current_mentions = [
                (s + offset if s > start else s,
                 e + offset if e > start else e, m, et)
                for s, e, m, et in current_mentions
            ]
            n_rep += 1
        if n_rep > 0:
            augmented.append({
                'id': str(sent['id']) + '_aug' + str(aug_idx),
                'tokens': new_tokens, 'tags': new_tags
            })
    return augmented

def run_cosiner_k(sentences, rankings, k):
    """COSINER tanpa imbalance-aware, dengan K yang divariasikan."""
    all_aug = []
    for sent in sentences:
        n_pos = sum(1 for t in sent['tags'] if t != 'O')
        if n_pos == 0: continue
        augs = augment_sentence(sent, rankings, k)
        all_aug.extend(augs)
    return all_aug

# === CELL BREAK ===
# --- MODEL & TRAINING ---
class NERDatasetBERT(Dataset):
    def __init__(self, sentences, tokenizer, tag2idx, max_len=MAX_SEQ_LEN):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.tag2idx   = tag2idx
        self.max_len   = max_len

    def __len__(self): return len(self.sentences)

    def __getitem__(self, idx):
        sent   = self.sentences[idx]
        tokens, tags = sent['tokens'], sent['tags']
        encoding = self.tokenizer(
            tokens, is_split_into_words=True,
            max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        word_ids = encoding.word_ids()
        aligned  = []; prev = None
        for wid in word_ids:
            if wid is None: aligned.append(-100)
            elif wid != prev:
                aligned.append(self.tag2idx[tags[wid]] if wid < len(tags) else -100)
            else:
                if wid < len(tags):
                    tag = tags[wid]
                    if tag.startswith('B-'): aligned.append(self.tag2idx['I-' + tag[2:]])
                    else: aligned.append(self.tag2idx[tag])
                else: aligned.append(-100)
            prev = wid
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(aligned, dtype=torch.long),
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            labels = batch['labels']
            preds  = model(input_ids=ids, attention_mask=mask).logits.argmax(dim=-1).cpu().tolist()
            for ps, ls in zip(preds, labels.tolist()):
                pt, tt = [], []
                for p, l in zip(ps, ls):
                    if l != -100:
                        pt.append(IDX2TAG.get(p, 'O'))
                        tt.append(IDX2TAG.get(l, 'O'))
                if tt: y_true.append(tt); y_pred.append(pt)
    return seqeval_f1(y_true, y_pred), seqeval_p(y_true, y_pred), seqeval_r(y_true, y_pred)

def train_single(train_sents, val_sents, test_sents, seed, label=""):
    set_seed(seed)
    train_loader = DataLoader(NERDatasetBERT(train_sents, tokenizer, TAG2IDX),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(NERDatasetBERT(val_sents,   tokenizer, TAG2IDX),
                              batch_size=BATCH_SIZE*4, shuffle=False)
    test_loader  = DataLoader(NERDatasetBERT(test_sents,  tokenizer, TAG2IDX),
                              batch_size=BATCH_SIZE*4, shuffle=False)

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_TAGS
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 0, len(train_loader) * NUM_EPOCHS
    )

    best_val_f1, best_state, best_epoch = 0.0, None, 0
    t0 = time.time()

    for epoch in range(NUM_EPOCHS):
        model.train(); total_loss = 0
        for batch in train_loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss   = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        vf1, vp, vr = evaluate_model(model, val_loader)
        print(f"    [{label}] Epoch {epoch+1}/{NUM_EPOCHS}: loss={avg_loss:.4f}, val_F1={vf1:.4f}")

        if vf1 > best_val_f1:
            best_val_f1 = vf1
            best_state  = deepcopy(model.state_dict())
            best_epoch  = epoch + 1

    if best_state:
        model.load_state_dict(best_state)
    tf1, tp, tr = evaluate_model(model, test_loader)
    elapsed = time.time() - t0

    print(f"    [{label}] >> Test: F1={tf1:.4f}, P={tp:.4f}, R={tr:.4f} | "
          f"Best val_F1={best_val_f1:.4f} @ epoch {best_epoch} | Time={elapsed:.1f}s")
    del model; torch.cuda.empty_cache()

    return {
        'val_f1': best_val_f1,
        'test_f1': tf1, 'test_p': tp, 'test_r': tr,
        'best_epoch': best_epoch, 'time': elapsed,
    }

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [5]:
# --- LOAD DATA ---
print("="*65)
print(f"LOADING NCBI DATA — {FEW_SHOT_LABEL}")
print("="*65)

DATASET_PATH = '/kaggle/input/datasets/wantodbz/ncbi-disease'
train_df = pd.read_csv(os.path.join(DATASET_PATH, 'train.csv'))
val_df   = pd.read_csv(os.path.join(DATASET_PATH, 'validation.csv'))
test_df  = pd.read_csv(os.path.join(DATASET_PATH, 'test.csv'))

train_sentences = parse_sentences(train_df)
val_sentences   = parse_sentences(val_df)
test_sentences  = parse_sentences(test_df)

fs_train = sample_few_shot(train_sentences, FEW_SHOT_FRACTION)
n_ent    = sum(1 for s in fs_train for t in s['tags'] if t.startswith('B-'))
print(f"Few-shot {FEW_SHOT_LABEL}: {len(fs_train)} sentences, {n_ent} entities")
print(f"Val: {len(val_sentences)} | Test: {len(test_sentences)}")

LOADING NCBI DATA — 2%
Few-shot 2%: 107 sentences, 89 entities
Val: 919 | Test: 938


In [6]:
# --- BUILD EMBEDDINGS (SEKALI) ---
print("\n" + "="*65)
print("BUILDING ENTITY EMBEDDINGS (dibangun sekali untuk semua K)")
print("="*65)

mention_contexts, mention_types = extract_mentions_from_sentences(fs_train)
print(f"Unique mentions: {len(mention_contexts)}")
embeddings = build_entity_embeddings(mention_contexts)
rankings   = compute_similarity_rankings(embeddings, mention_types)
print("Done: Embeddings & rankings siap.")


BUILDING ENTITY EMBEDDINGS (dibangun sekali untuk semua K)
Unique mentions: 69


pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Done: Embeddings & rankings siap.


In [7]:
# === CELL BREAK ===
# --- GRID SEARCH LOOP ---
print("\n" + "="*65)
print(f"GRID SEARCH COSINER K — NCBI, seed={GRID_SEED}")
print("="*65)

gs_records = []

for k in K_GRID:
    print(f"\n{'─'*55}")
    print(f"K = {k}")
    print(f"{'─'*55}")

    aug_sents = run_cosiner_k(fs_train, rankings, k)
    combined  = fs_train + aug_sents
    n_aug     = len(aug_sents)
    n_total   = len(combined)
    aug_ratio = n_aug / max(len(fs_train), 1)

    print(f"  Original  : {len(fs_train)} sentences")
    print(f"  Augmented : {n_aug} sentences")
    print(f"  Combined  : {n_total} sentences (aug_ratio={aug_ratio:.2f}x)")

    res = train_single(combined, val_sentences, test_sentences,
                       seed=GRID_SEED, label=f"COS_K{k}")

    gs_records.append({
        'K'          : k,
        'n_original' : len(fs_train),
        'n_augmented': n_aug,
        'n_combined' : n_total,
        'aug_ratio'  : round(aug_ratio, 2),
        'val_f1'     : round(res['val_f1'], 4),
        'test_f1'    : round(res['test_f1'], 4),
        'test_p'     : round(res['test_p'], 4),
        'test_r'     : round(res['test_r'], 4),
        'best_epoch' : res['best_epoch'],
        'time_sec'   : round(res['time'], 1),
    })
    print(f"  val_F1={res['val_f1']:.4f} | test_F1={res['test_f1']:.4f}")

# === CELL BREAK ===
# --- SUMMARY ---
print("\n" + "="*65)
print("GRID SEARCH SUMMARY — NCBI COSINER K")
print("="*65)

df_gs    = pd.DataFrame(gs_records).sort_values('val_f1', ascending=False)
best_row = df_gs.iloc[0]
best_k   = int(best_row['K'])

print(df_gs[['K', 'n_augmented', 'n_combined', 'aug_ratio',
             'val_f1', 'test_f1', 'test_p', 'test_r',
             'best_epoch', 'time_sec']].to_string(index=False))
print(f"\n★ K terbaik (berdasar val_F1): K={best_k}")
print(f"  val_F1={best_row['val_f1']:.4f} | test_F1={best_row['test_f1']:.4f}")

df_gs.to_csv(os.path.join(OUTPUT_DIR, 'gridsearch_k_ncbi_results.csv'), index=False)
print(f"Hasil disimpan → {OUTPUT_DIR}/gridsearch_k_ncbi_results.csv")

# === CELL BREAK ===
# --- VISUALISASI ---
df_plot = pd.DataFrame(gs_records).sort_values('K')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Grid Search COSINER K — NCBI Disease, seed={GRID_SEED}, {FEW_SHOT_LABEL}',
             fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(df_plot['K'], df_plot['val_f1'],  'o-', color='#3498db',
        linewidth=2, markersize=8, label='Val F1')
ax.plot(df_plot['K'], df_plot['test_f1'], 's--', color='#e74c3c',
        linewidth=2, markersize=8, label='Test F1')
ax.axvline(best_k, color='#27ae60', linestyle=':', linewidth=1.5,
           label=f'Best K={best_k}')
ax.set_xlabel('K (AUG_TOP_K)'); ax.set_ylabel('F1 Score')
ax.set_title('Val & Test F1 vs K')
ax.set_xticks(K_GRID)
ax.legend(); ax.grid(alpha=0.3)
for _, row in df_plot.iterrows():
    ax.annotate(f"{row['val_f1']:.3f}", (row['K'], row['val_f1']),
                textcoords="offset points", xytext=(0, 8),
                ha='center', fontsize=8)

ax = axes[1]
ax.bar(df_plot['K'].astype(str), df_plot['n_augmented'],
       color='#e67e22', edgecolor='black', linewidth=0.5, alpha=0.8,
       label='Augmented')
ax.bar(df_plot['K'].astype(str), df_plot['n_original'],
       color='#95a5a6', edgecolor='black', linewidth=0.5, alpha=0.8,
       label='Original', bottom=0)
ax.set_xlabel('K'); ax.set_ylabel('Jumlah Sentences')
ax.set_title('Volume Data per K')
ax.legend(); ax.grid(axis='y', alpha=0.3)

ax = axes[2]
prf_data = df_plot[['K', 'test_p', 'test_r', 'test_f1']].set_index('K')
im = ax.imshow(prf_data.T.values, aspect='auto', cmap='YlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(prf_data)))
ax.set_xticklabels(prf_data.index.tolist())
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Precision', 'Recall', 'F1'])
ax.set_title('Test P/R/F1 Heatmap per K')
ax.set_xlabel('K')
for i in range(3):
    for j in range(len(prf_data)):
        ax.text(j, i, f"{prf_data.values[j, i]:.3f}",
                ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gridsearch_k_ncbi_plot.png'),
            dpi=300, bbox_inches='tight')
plt.show(); plt.close()
print("Plot disimpan.")

# === CELL BREAK ===
print("\n" + "="*65)
print("GRID SEARCH SELESAI — NCBI COSINER K")
print("="*65)
print(f"\nK terbaik  : {best_k}")
print(f"Val F1     : {best_row['val_f1']:.4f}")
print(f"Test F1    : {best_row['test_f1']:.4f}")
print(f"Test P     : {best_row['test_p']:.4f}")
print(f"Test R     : {best_row['test_r']:.4f}")
print(f"\nOutput dir : {OUTPUT_DIR}")



GRID SEARCH COSINER K — NCBI, seed=64

───────────────────────────────────────────────────────
K = 1
───────────────────────────────────────────────────────
  Original  : 107 sentences
  Augmented : 52 sentences
  Combined  : 159 sentences (aug_ratio=0.49x)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    [COS_K1] Epoch 1/5: loss=0.3913, val_F1=0.5115
    [COS_K1] Epoch 2/5: loss=0.0708, val_F1=0.6086
    [COS_K1] Epoch 3/5: loss=0.0195, val_F1=0.6421
    [COS_K1] Epoch 4/5: loss=0.0062, val_F1=0.6481
    [COS_K1] Epoch 5/5: loss=0.0048, val_F1=0.6582
    [COS_K1] >> Test: F1=0.6667, P=0.6585, R=0.6751 | Best val_F1=0.6582 @ epoch 5 | Time=270.3s
  val_F1=0.6582 | test_F1=0.6667

───────────────────────────────────────────────────────
K = 2
───────────────────────────────────────────────────────
  Original  : 107 sentences
  Augmented : 104 sentences
  Combined  : 211 sentences (aug_ratio=0.97x)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    [COS_K8] Epoch 1/5: loss=0.1651, val_F1=0.6157
    [COS_K8] Epoch 2/5: loss=0.0051, val_F1=0.6584
    [COS_K8] Epoch 3/5: loss=0.0008, val_F1=0.6552
    [COS_K8] Epoch 4/5: loss=0.0008, val_F1=0.6332
    [COS_K8] Epoch 5/5: loss=0.0004, val_F1=0.6399
    [COS_K8] >> Test: F1=0.6609, P=0.6454, R=0.6771 | Best val_F1=0.6584 @ epoch 2 | Time=467.6s
  val_F1=0.6584 | test_F1=0.6609

───────────────────────────────────────────────────────
K = 9
───────────────────────────────────────────────────────
  Original  : 107 sentences
  Augmented : 468 sentences
  Combined  : 575 sentences (aug_ratio=4.37x)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    [COS_K9] Epoch 1/5: loss=0.1619, val_F1=0.5891
    [COS_K9] Epoch 2/5: loss=0.0051, val_F1=0.5980
    [COS_K9] Epoch 3/5: loss=0.0017, val_F1=0.6402
    [COS_K9] Epoch 4/5: loss=0.0006, val_F1=0.6549
    [COS_K9] Epoch 5/5: loss=0.0002, val_F1=0.6376
    [COS_K9] >> Test: F1=0.6762, P=0.6613, R=0.6918 | Best val_F1=0.6549 @ epoch 4 | Time=494.1s
  val_F1=0.6549 | test_F1=0.6762

GRID SEARCH SUMMARY — NCBI COSINER K
 K  n_augmented  n_combined  aug_ratio  val_f1  test_f1  test_p  test_r  best_epoch  time_sec
 5          260         367       2.43  0.6800   0.6711  0.6573  0.6855           3     387.8
 7          364         471       3.40  0.6638   0.6900  0.6696  0.7117           3     440.7
 4          208         315       1.94  0.6586   0.6810  0.6501  0.7149           3     361.2
 8          416         523       3.89  0.6584   0.6609  0.6454  0.6771           2     467.6
 1           52         159       0.49  0.6582   0.6667  0.6585  0.6751           5     270.3
 9          4